# 03 — EDA and Feature Engineering: PD Model

This notebook covers the exploratory data analysis and feature engineering for the PD model features. Outliers are treated, categorical features are reviewed, and the cleaned dataframe is saved for training.

**Input:** `data/processed/checkpoint_DataFrame_for_EDA.pkl`, `data/processed/features_PD_for_EDA.pkl`  
**Output:** `data/processed/checkpoint_df_PD_training.pkl`, `data/processed/features_PD_num_for_training.pkl`, `data/processed/features_PD_cat_for_training.pkl`

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

from feature_plot import plot_boxplots_list, plot_categoricals
from feature_statistics import dataset_audit, outlier_summary
from feature_selection import compute_plot_correlation_matrix
from clean_features_PD import clean_features_PD

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

with open("data/processed/features_PD_for_EDA.pkl", "rb") as f:
    features_PD = pickle.load(f)

# Isolated PD dataframe — df_master_features is never modified
df_PD = df_master_features[["Target_Variable_Default"] + features_PD].copy()

print(f"df_PD shape: {df_PD.shape}")
print(f"Features: {features_PD}")

## 2. Initial Audit

In [ ]:
audit_PD = dataset_audit(df_PD[features_PD])
audit_PD

## 3. Numeric Features

### 3.1 Distribution Overview

In [ ]:
# Identify numeric features — exclude is_directpay_PD (binary, treated as categorical)
num_features_PD = df_PD[features_PD].select_dtypes(include=[np.number]).columns.tolist()
num_features_PD.remove("is_directpay_PD")

plot_boxplots_list(df_PD, num_features_PD, cols=4)

### 3.2 Outlier Summary

In [ ]:
outlier_summary_PD = outlier_summary(df_PD, num_features_PD)
outlier_summary_PD

### 3.3 Treatment Decisions

Detailed justification for each feature treatment is documented in `docs/pd_feature_treatment.md`. Summary:

| Feature | Treatment |
|---|---|
| `interest_rate` | None |
| `monthly_payment` | None |
| `annual_income` | Cap p99, log1p if skew > 2 post-capping |
| `used_credit_share` | Cap at 100 (economic ceiling) |
| `num_inq_in_6mths` | None |
| `num_rev_trades_op_in_12mths` | None |
| `num_inq_in_12mths` | Cap at p99 = 11 |
| `max_bal_owed` | Cap p99 |
| `dept_paym_income_ratio` | Replace negatives with 0 |
| `bal_to_cred_lim` | Cap at 100, impute NaNs with median post-capping |
| `emp_length` | None |
| `earliest_cr_line_year` | Cap at p1 (lower tail errors) |

In [ ]:
df_PD_num_clean = clean_features_PD(df_PD)

In [ ]:
# Post-treatment audit
audit_PD_num_clean = dataset_audit(df_PD_num_clean[num_features_PD])
audit_PD_num_clean

In [ ]:
# Post-treatment boxplots
plot_boxplots_list(df_PD_num_clean, num_features_PD, cols=4)

In [ ]:
# Post-treatment outlier summary
outlier_summary_PD_num_clean = outlier_summary(df_PD_num_clean, num_features_PD)
outlier_summary_PD_num_clean

## 4. Categorical Features

In [ ]:
cat_features_PD = df_PD_num_clean.select_dtypes(include=["object"]).columns.tolist()
plot_categoricals(df_PD_num_clean, cat_features_PD, figsize_per_plot=(6, 4))

In [ ]:
# home_ownership_status: OTHER (0.07%) grouped into RENT — closest economic profile
df_PD_num_clean["home_ownership_status"] = df_PD_num_clean["home_ownership_status"].replace("OTHER", "RENT")

# Confirm distribution post-grouping
plot_categoricals(df_PD_num_clean, cat_features_PD, figsize_per_plot=(6, 4))

## 5. Post-Treatment Correlation Check

In [ ]:
corr_matrix_PD = compute_plot_correlation_matrix(df_PD_num_clean)

# used_credit_share vs bal_to_cred_lim: 0.67 — below 0.85 threshold, no action required
df_PD_num_clean[["used_credit_share", "bal_to_cred_lim"]].corr()

In [ ]:
df_PD_clean = df_PD_num_clean.copy()

## 6. Checkpoint: Save PD Training Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_df_PD_training.pkl", "wb") as f:
    pickle.dump({"df_PD_clean": df_PD_clean}, f)

with open("data/processed/features_PD_num_for_training.pkl", "wb") as f:
    pickle.dump(num_features_PD, f)

with open("data/processed/features_PD_cat_for_training.pkl", "wb") as f:
    pickle.dump(cat_features_PD, f)

print(f"Saved. Shape: {df_PD_clean.shape}")

In [ ]:
# Load checkpoint (skip re-running)
with open("data/processed/checkpoint_df_PD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_PD_clean = data["df_PD_clean"]

with open("data/processed/features_PD_num_for_training.pkl", "rb") as f:
    num_features_PD = pickle.load(f)

with open("data/processed/features_PD_cat_for_training.pkl", "rb") as f:
    cat_features_PD = pickle.load(f)